# Chapter 4: Data Pipeline for RAG

**From *AI Enterprise Architect***

Build a complete RAG data pipeline: ingest documents, chunk text, generate embeddings,
store in a vector database, and query. This notebook demonstrates how chunking strategy
affects retrieval quality — the #1 lever for RAG performance.

## What You'll Learn
- How to prepare documents for a RAG pipeline
- Three chunking strategies: fixed-size, sentence-based, and overlapping windows
- How to generate and store embeddings with metadata
- How to query a vector store and compare retrieval quality across strategies

## Prerequisites
- An OpenAI API key (for generating embeddings)
- Basic Python knowledge

In [ ]:
# Install dependencies
!pip install -q openai numpy tiktoken

## 1. Configuration

Set your OpenAI API key as an environment variable. In Google Colab, you can use
the Secrets panel (key icon in the left sidebar) or set it directly:

```python
import os
os.environ["OPENAI_API_KEY"] = "sk-..."
```

In [ ]:
import os
import json
import time
import numpy as np
from datetime import datetime
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Tuple

from openai import OpenAI
import tiktoken

# Retrieve API key from environment
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError(
        "Please set the OPENAI_API_KEY environment variable. "
        "In Colab, use the Secrets panel or: os.environ['OPENAI_API_KEY'] = 'sk-...'"
    )

client = OpenAI(api_key=OPENAI_API_KEY)
EMBEDDING_MODEL = "text-embedding-3-small"
print("Configuration loaded successfully.")

## 2. Sample Documents

We simulate PDF content as plain-text strings. In production, you would use a PDF
parser (e.g., PyMuPDF, pdfplumber) to extract text. Here we use three documents
covering different enterprise architecture topics so we can test retrieval relevance.

In [ ]:
# Simulated document corpus — three documents on enterprise architecture topics.
# Each document is long enough to produce multiple chunks under every strategy.

DOCUMENTS = {
    "enterprise_architecture_basics.pdf": (
        "Enterprise architecture (EA) is a discipline for proactively and holistically "
        "leading enterprise responses to disruptive forces by identifying and analyzing "
        "the execution of change toward desired business vision and outcomes. EA delivers "
        "value by presenting business and IT leaders with signature-ready recommendations "
        "for adjusting policies and projects to achieve targeted business outcomes that "
        "capitalize on relevant business disruptions. The TOGAF framework is one of the "
        "most widely adopted EA frameworks. It provides a systematic approach for designing, "
        "planning, implementing, and governing enterprise information architecture. "
        "TOGAF divides architecture into four domains: business architecture, data architecture, "
        "application architecture, and technology architecture. Each domain addresses a "
        "different layer of the enterprise stack. Business architecture defines the business "
        "strategy, governance, organization, and key business processes. Data architecture "
        "describes the structure of an organization's logical and physical data assets and "
        "data management resources. Application architecture provides a blueprint for the "
        "individual applications to be deployed, their interactions, and their relationships "
        "to the core business processes. Technology architecture describes the logical software "
        "and hardware capabilities required to support the deployment of business, data, and "
        "application services. A mature EA practice enables organizations to make better "
        "decisions about technology investments, reduce redundancy, improve interoperability, "
        "and align IT strategy with business goals."
    ),
    "ai_integration_patterns.pdf": (
        "Integrating AI into enterprise systems requires careful architectural planning. "
        "The most common patterns include the sidecar pattern, where an AI service runs "
        "alongside an existing application and augments its capabilities without modifying "
        "the core codebase. The gateway pattern routes all AI requests through a central "
        "service that handles authentication, rate limiting, model selection, and logging. "
        "This pattern is essential for governance and cost control. The event-driven pattern "
        "uses message queues or event streams to trigger AI processing asynchronously. This "
        "is ideal for batch processing, document analysis, and scenarios where real-time "
        "response is not critical. RAG, or retrieval-augmented generation, is an integration "
        "pattern that combines a retrieval system with a generative model. Documents are "
        "chunked, embedded, and stored in a vector database. At query time, relevant chunks "
        "are retrieved and included in the prompt to ground the model's response in factual "
        "data. RAG dramatically reduces hallucination and makes AI systems more trustworthy "
        "for enterprise use. The quality of a RAG system depends heavily on the chunking "
        "strategy, embedding model choice, and retrieval algorithm. Poorly chunked documents "
        "lead to irrelevant retrievals and degraded answer quality. The orchestration pattern "
        "uses a central controller to coordinate multiple AI agents or models, each specialized "
        "for a particular subtask. This pattern is emerging in complex workflows like "
        "contract analysis, supply chain optimization, and multi-step reasoning tasks. "
        "Choosing the right integration pattern depends on latency requirements, cost "
        "constraints, data sensitivity, and the maturity of the organization's AI platform."
    ),
    "data_governance_for_ai.pdf": (
        "Data governance is the foundation of any successful AI initiative. Without high-quality, "
        "well-governed data, AI models will produce unreliable outputs. Key principles of data "
        "governance for AI include data lineage tracking, which records where data comes from, "
        "how it has been transformed, and where it is used. Data quality metrics such as "
        "completeness, accuracy, consistency, and timeliness must be continuously monitored. "
        "Access control policies ensure that sensitive data is only available to authorized "
        "users and systems. This is especially important when AI systems process personally "
        "identifiable information or protected health information. Data catalogs provide a "
        "searchable inventory of all data assets in the organization, making it easier for "
        "data scientists and engineers to discover and use relevant datasets. Master data "
        "management ensures consistency of key business entities like customers, products, "
        "and locations across all systems. For AI-specific governance, organizations should "
        "also track training data provenance, model bias metrics, and fairness indicators. "
        "The EU AI Act and similar regulations are making data governance for AI not just a "
        "best practice but a legal requirement. Organizations that invest in data governance "
        "early will have a significant advantage when scaling their AI initiatives. A data "
        "governance council should include representatives from IT, legal, compliance, and "
        "business units to ensure balanced decision-making. Metadata management, data "
        "classification, and retention policies are all critical components of a comprehensive "
        "data governance program."
    ),
}

print(f"Loaded {len(DOCUMENTS)} documents:")
for name, text in DOCUMENTS.items():
    print(f"  - {name}: {len(text)} characters")

## 3. Chunking Strategies

Chunking is the most impactful decision in a RAG pipeline. We implement three strategies:

1. **Fixed-size (token-based)**: Split every 200 tokens. Simple but may cut mid-sentence.
2. **Sentence-based**: Split on sentence boundaries. Preserves semantic coherence.
3. **Overlapping windows**: 500-token windows with 100-token overlap. Captures cross-boundary context.

Each chunk carries metadata (source document, chunk index, strategy, timestamp).

In [ ]:
@dataclass
class Chunk:
    """A text chunk with metadata for the RAG pipeline."""
    text: str
    source_doc: str
    chunk_index: int
    strategy: str
    timestamp: str = field(default_factory=lambda: datetime.utcnow().isoformat())
    token_count: int = 0


# We use tiktoken to count tokens consistently with the OpenAI API.
enc = tiktoken.get_encoding("cl100k_base")


def count_tokens(text: str) -> int:
    """Count the number of tokens in a text string."""
    return len(enc.encode(text))


# --- Strategy 1: Fixed-size token-based chunking ---
def chunk_fixed_size(text: str, source: str, max_tokens: int = 200) -> List[Chunk]:
    """
    Split text into chunks of exactly max_tokens tokens.
    Simple and fast, but may break mid-sentence.
    """
    tokens = enc.encode(text)
    chunks = []
    for i in range(0, len(tokens), max_tokens):
        chunk_tokens = tokens[i:i + max_tokens]
        chunk_text = enc.decode(chunk_tokens)
        chunks.append(Chunk(
            text=chunk_text,
            source_doc=source,
            chunk_index=len(chunks),
            strategy="fixed_size_200",
            token_count=len(chunk_tokens),
        ))
    return chunks


# --- Strategy 2: Sentence-based chunking ---
def chunk_by_sentence(text: str, source: str) -> List[Chunk]:
    """
    Split text on sentence boundaries (periods followed by a space).
    Preserves semantic coherence within each chunk.
    """
    # Simple sentence splitter — split on '. ' to keep sentence structure.
    raw_sentences = text.split(". ")
    chunks = []
    for i, sentence in enumerate(raw_sentences):
        # Re-add the period that was removed by split, except for the last segment.
        if i < len(raw_sentences) - 1:
            sentence = sentence + "."
        sentence = sentence.strip()
        if not sentence:
            continue
        chunks.append(Chunk(
            text=sentence,
            source_doc=source,
            chunk_index=len(chunks),
            strategy="sentence_based",
            token_count=count_tokens(sentence),
        ))
    return chunks


# --- Strategy 3: Overlapping windows ---
def chunk_overlapping(text: str, source: str, window: int = 500, overlap: int = 100) -> List[Chunk]:
    """
    Split text into windows of `window` tokens with `overlap` token overlap.
    Captures context that spans chunk boundaries.
    """
    tokens = enc.encode(text)
    step = window - overlap
    chunks = []
    for i in range(0, len(tokens), step):
        chunk_tokens = tokens[i:i + window]
        if not chunk_tokens:
            break
        chunk_text = enc.decode(chunk_tokens)
        chunks.append(Chunk(
            text=chunk_text,
            source_doc=source,
            chunk_index=len(chunks),
            strategy="overlapping_500_100",
            token_count=len(chunk_tokens),
        ))
        # Stop if we've consumed all tokens.
        if i + window >= len(tokens):
            break
    return chunks


print("Chunking functions defined.")

### Apply All Three Strategies

We chunk every document with each strategy and compare the results.

In [ ]:
# Apply all three chunking strategies to all documents.
chunking_strategies = {
    "fixed_size_200": chunk_fixed_size,
    "sentence_based": chunk_by_sentence,
    "overlapping_500_100": chunk_overlapping,
}

# Store chunks organized by strategy name.
all_chunks: Dict[str, List[Chunk]] = {}

for strategy_name, chunk_fn in chunking_strategies.items():
    strategy_chunks = []
    for doc_name, doc_text in DOCUMENTS.items():
        strategy_chunks.extend(chunk_fn(doc_text, doc_name))
    all_chunks[strategy_name] = strategy_chunks

# Summary
print("Chunking results:")
print(f"{'Strategy':<25} {'Chunks':>6}  {'Avg Tokens':>10}  {'Min':>4}  {'Max':>4}")
print("-" * 60)
for strategy_name, chunks in all_chunks.items():
    token_counts = [c.token_count for c in chunks]
    print(
        f"{strategy_name:<25} {len(chunks):>6}  "
        f"{np.mean(token_counts):>10.1f}  "
        f"{min(token_counts):>4}  {max(token_counts):>4}"
    )

## 4. Generate Embeddings

We use OpenAI's `text-embedding-3-small` model to embed each chunk. The embeddings
are stored alongside the chunk metadata so we can retrieve them later.

In [ ]:
def get_embeddings(texts: List[str], model: str = EMBEDDING_MODEL) -> List[List[float]]:
    """
    Call the OpenAI embeddings API for a batch of texts.
    Returns a list of embedding vectors.
    """
    response = client.embeddings.create(input=texts, model=model)
    return [item.embedding for item in response.data]


# Embed all chunks for each strategy.
# We store embeddings in a parallel dict keyed by strategy name.
all_embeddings: Dict[str, np.ndarray] = {}

for strategy_name, chunks in all_chunks.items():
    print(f"Embedding {len(chunks)} chunks for strategy: {strategy_name} ...")
    texts = [c.text for c in chunks]
    embeddings = get_embeddings(texts)
    all_embeddings[strategy_name] = np.array(embeddings)
    print(f"  Done. Embedding shape: {all_embeddings[strategy_name].shape}")

print("\nAll embeddings generated.")

## 5. Simple Vector Store

We build a lightweight vector store using NumPy cosine similarity. In production,
you'd use a dedicated vector database (Pinecone, Weaviate, Qdrant, pgvector, etc.),
but the math is the same.

In [ ]:
class SimpleVectorStore:
    """A minimal vector store using cosine similarity for retrieval."""

    def __init__(self, chunks: List[Chunk], embeddings: np.ndarray):
        self.chunks = chunks
        # Normalize embeddings for cosine similarity (dot product of unit vectors).
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        self.embeddings = embeddings / norms

    def query(self, query_embedding: np.ndarray, top_k: int = 3) -> List[Tuple[Chunk, float]]:
        """
        Retrieve the top_k most similar chunks to the query embedding.
        Returns a list of (chunk, similarity_score) tuples.
        """
        # Normalize query embedding.
        query_norm = query_embedding / np.linalg.norm(query_embedding)
        # Cosine similarity = dot product of normalized vectors.
        similarities = self.embeddings @ query_norm
        # Get top-k indices.
        top_indices = np.argsort(similarities)[::-1][:top_k]
        return [(self.chunks[i], float(similarities[i])) for i in top_indices]


# Build a vector store for each chunking strategy.
stores: Dict[str, SimpleVectorStore] = {}
for strategy_name in all_chunks:
    stores[strategy_name] = SimpleVectorStore(
        all_chunks[strategy_name],
        all_embeddings[strategy_name],
    )

print(f"Built {len(stores)} vector stores (one per chunking strategy).")

## 6. Query and Compare Retrieval Quality

We run **three queries** against each vector store and compare:
- Which chunks are retrieved
- The relevance scores (cosine similarity)
- Whether the retrieved chunks actually contain the answer

This is the core experiment — see how chunking strategy changes what the model "sees."

In [ ]:
# Three test queries covering different topics from our documents.
QUERIES = [
    "What are the four domains of the TOGAF framework?",
    "How does RAG reduce hallucination in enterprise AI systems?",
    "What data governance practices are required for AI compliance?",
]

TOP_K = 3  # Number of chunks to retrieve per query.

# Embed all queries at once.
query_embeddings = get_embeddings(QUERIES)

# Run each query against each strategy's vector store.
for q_idx, query in enumerate(QUERIES):
    print("=" * 80)
    print(f"QUERY {q_idx + 1}: {query}")
    print("=" * 80)

    q_emb = np.array(query_embeddings[q_idx])

    for strategy_name, store in stores.items():
        results = store.query(q_emb, top_k=TOP_K)
        print(f"\n  Strategy: {strategy_name}")
        print(f"  {'-' * 60}")
        for rank, (chunk, score) in enumerate(results, 1):
            # Truncate text for display.
            preview = chunk.text[:120].replace("\n", " ") + "..."
            print(f"  [{rank}] Score: {score:.4f} | Source: {chunk.source_doc} | "
                  f"Chunk #{chunk.chunk_index} ({chunk.token_count} tokens)")
            print(f"      {preview}")
    print()

## 7. Analysis: How Chunk Size Affects Answer Quality

Let's quantify the differences. We compute average retrieval scores and see which
strategies return chunks from the most relevant source documents.

In [ ]:
# Expected best source document for each query (ground truth for evaluation).
EXPECTED_SOURCES = [
    "enterprise_architecture_basics.pdf",   # TOGAF question
    "ai_integration_patterns.pdf",           # RAG question
    "data_governance_for_ai.pdf",            # Governance question
]

print(f"{'Strategy':<25} {'Avg Score':>10} {'Source Accuracy':>16}")
print("-" * 55)

for strategy_name, store in stores.items():
    scores = []
    correct_source = 0
    total = 0

    for q_idx, query in enumerate(QUERIES):
        q_emb = np.array(query_embeddings[q_idx])
        results = store.query(q_emb, top_k=TOP_K)
        for chunk, score in results:
            scores.append(score)
            if chunk.source_doc == EXPECTED_SOURCES[q_idx]:
                correct_source += 1
            total += 1

    avg_score = np.mean(scores)
    source_accuracy = correct_source / total
    print(f"{strategy_name:<25} {avg_score:>10.4f} {source_accuracy:>15.1%}")

print("\nSource accuracy = fraction of retrieved chunks from the expected document.")
print("Higher is better — it means the strategy retrieves more relevant content.")

## 8. Inspect Chunk Metadata

Every chunk carries metadata that would be stored alongside the embedding in a
production vector database. Let's inspect a few examples.

In [ ]:
# Show metadata for the first 3 chunks of each strategy.
for strategy_name, chunks in all_chunks.items():
    print(f"\nStrategy: {strategy_name}")
    print("-" * 50)
    for chunk in chunks[:3]:
        metadata = {
            "source_doc": chunk.source_doc,
            "chunk_index": chunk.chunk_index,
            "strategy": chunk.strategy,
            "token_count": chunk.token_count,
            "timestamp": chunk.timestamp,
        }
        print(json.dumps(metadata, indent=2))

## Key Takeaways

1. **Chunking strategy is the #1 lever for RAG quality.** The same query against the
   same documents can return very different results depending on how you chunk.

2. **Sentence-based chunking** preserves semantic coherence but may produce very small
   or very large chunks depending on writing style.

3. **Fixed-size chunking** is simple and predictable but may split concepts across
   chunk boundaries, losing context.

4. **Overlapping windows** capture cross-boundary context and are often the best
   default for production RAG systems.

5. **Metadata matters.** Storing source, chunk index, and timestamps alongside
   embeddings enables filtering, debugging, and audit trails.

### For Enterprise Architects

When designing a RAG architecture, treat chunking as a first-class architectural
decision. Document your chunking strategy in your Architecture Decision Records (ADRs)
and plan for A/B testing different strategies in production. The retrieval layer is
where most RAG quality problems originate — not in the LLM itself.